In [ ]:

class FakePeData:
    def __init__(self, email_domain: str = "magnifyfirm.com", static_staff: dict | None = None):

        """Static Staff is list of default staff users that can be used for logging in, the rest are randomly generated"""
        # self.phone_number_sub = phone_number
        # print(self.phone_number_sub.Provider.formats)
        self.fake = Faker(locale="en_US")
        self.email_domain = email_domain
        self.static_staff = static_staff
        # class RestrictedPhoneNumber(fake.Provider):
        #     def phone_number(self, custom_formats: list[str] = None) -> str:
        #         if custom_formats:
        #             return self.numerify(self.random_element(custom_formats))
        #         return self.numerify(self.random_element(self.formats))
                
        self.fake.add_provider(person)
        self.fake.add_provider(address)
        self.fake.add_provider(company)
        self.fake.add_provider(phone_number)
        self.fake.add_provider(date_time)

    def fake_contact(self) -> dict:
        address = self.fake.street_address()
        city = self.fake.city()
        state = self.fake.state()
        zipcode = self.fake.zipcode()
        phone = self.fake.phone_number(custom_formats=["($##)$##-####", "($##)$##-####"])
        email = self.fake.email()
        return {
            "ADDRESS": address,
            "CITY": city,
            "STATE": state,
            "ZIPCODE": zipcode,
            "PHONE": phone,
            "EMAIL": email,
        }

    def fake_staff(self, staff_obj: pd.Series) -> dict:
        staff_index: int = int(staff_obj.STAFFINDEX)

        if staff_obj.SEX == "MALE":
            first_name = self.fake.first_name_male()
        else:
            first_name = self.fake.first_name_female()

        last_name: str = self.fake.last_name()
        employee: str = f"{first_name} {last_name}"
        employee_id: str = f"{employee} ({staff_index})"
        email: str = f"{first_name[0]}{last_name}@{self.email_domain}"

        contact: dict = self.fake_contact()
        contact.pop("EMAIL")

        return {
            "STAFFINDEX": staff_index,
            "FIRST_NAME": first_name,
            "LAST_NAME": last_name,
            "EMPLOYEE": employee,
            "EMPLOYEE_ID": employee_id,
            "STAFF_EMAIL": email,
            **contact
        }
    
    def fake_client(self, client_obj: pd.Series) -> dict:
        client = self.fake.company()
        billinggroup = f"{client} Group"
        
        return {
            "CONTINDEX": int(client_obj.CONTINDEX),
            "CLIENT": client,
            "BILLINGGROUP": billinggroup,
            **self.fake_contact()
        }

    # def __create_static_staff(self, staff_obj: dict) ->  dict:
        

    def generate_staff_table(self, staff_table_obj: pd.DataFrame) -> pd.DataFrame:
        fake_staff_list: list[dict] = [self.fake_staff(row) for row in staff_table_obj.itertuples()]
        fake_staff_df: pd.DataFrame = pd.DataFrame.from_records(fake_staff_list)

        if self.static_staff and isinstance(self.static_staff, list):
            for staff in self.static_staff:
                print(staff)
        elif self.static_staff and isinstance(self.static_staff, dict):
            print(self.static_staff)


        return fake_staff_df
    
    def generate_client_table(self, client_table_obj: pd.DataFrame) -> pd.DataFrame:
        fake_client_list: list[dict] = [self.fake_client(row) for row in client_table_obj.itertuples()]
        fake_client_df: pd.DataFrame = pd.DataFrame.from_records(fake_client_list)
        return fake_client_df
            
